# 2. The fair forecast arena
**45 minutes | 4 milestones | one dataset, one protocol**

Goal: compare seasonal naive, seasonal ARIMA, exponential smoothing, and
available Chronos-2 / TimesFM 2.5 / t0-gamma results on exactly the same
three forecast origins. Prophet is optional after the main session.

This notebook is self-contained. The last 42 days were reserved before
exploratory analysis in notebook 1. We are now allowed to reveal them
for evaluation, not use them to secretly tune model settings.

## RUN THIS — get the workshop files
**Colab:** upload the trusted workshop ZIP when asked (the one you get
from the repository's green **Code → Download ZIP** button, or from the
organizer). It contains one folder with the data and helpers. Each
notebook may have its own runtime. **Local:** start Jupyter from the
workshop folder.
This cell does not install anything or download model weights.

In [ ]:
from pathlib import Path
import os
import sys
import zipfile

here = Path.cwd()
found = next(here.glob("*/workshop.py"), None)  # an already extracted ZIP
if (here / "workshop.py").exists():
    ROOT = here
elif (here.parent / "workshop.py").exists():
    ROOT = here.parent
elif found:
    ROOT = found.parent
else:
    if "google.colab" not in sys.modules and not os.environ.get("COLAB_RELEASE_TAG"):
        raise FileNotFoundError("Extract the ZIP and start Jupyter in the workshop folder.")
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one workshop ZIP.")
    archive_path = Path(next(iter(uploaded)))
    with zipfile.ZipFile(archive_path) as archive:
        tops = {name.split("/")[0] for name in archive.namelist() if name.strip("/")}
        if len(tops) != 1:
            raise ValueError("The ZIP must contain exactly one top-level folder.")
        top = tops.pop()
        if top in ("", ".", "..") or Path(top).is_absolute():
            raise ValueError("Unsafe folder name in the ZIP.")
        for member in archive.infolist():
            destination = (here / member.filename).resolve()
            if not destination.is_relative_to((here / top).resolve()):
                raise ValueError("Unsafe path in the ZIP.")
        archive.extractall(here)
    ROOT = here / top
    if not (ROOT / "workshop.py").exists():
        raise ValueError("workshop.py not found: is this the workshop ZIP?")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Workshop files:", ROOT)

### SETUP ONLY — do this before the workshop
Set `INSTALL_CORE = True` if the core packages are missing, and
`INSTALL_FOUNDATION = True` to install the foundation-model packages.
You need the foundation packages to run the pretrained models yourself;
they are much larger than the core (see `SETUP.md`). Without them you
can still use the instructor's pre-computed forecasts. After
installation, restart the session if requested; rerun with both flags
`False`. A GPU is not necessary.
Read `SETUP.md` for downloads, CPU fallback, and troubleshooting.

In [ ]:
import subprocess

def run(command):
    # Run a command and show its output live, like a terminal.
    process = subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
    return process.wait()

INSTALL_CORE = False
INSTALL_FOUNDATION = False
for wanted, name in ((INSTALL_CORE, "core"), (INSTALL_FOUNDATION, "foundation")):
    if wanted:
        print(f"Installing {name} packages: this can take several minutes...")
        if run([sys.executable, "-m", "pip", "install", "-r", f"requirements/{name}.txt"]):
            raise RuntimeError(f"Installing {name} failed. Read the messages above.")
        print(f"Finished installing {name} packages.")

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, HTML
from workshop import (
    load_series, demo_split, make_model, validate_prediction,
    benchmark_series, splitter, run_benchmark, leaderboard, load_run,
    CONTEXT, HORIZON, N_FOLDS, SEASON, BASELINES, FOUNDATIONS,
)
if run([sys.executable, "scripts/check_environment.py"]):
    raise RuntimeError("The environment check failed. Read the messages above.")

# Plot helpers: one colour per entity, colour-blind-safe (Okabe-Ito).
pio.templates.default = "plotly_white"
BLUE, ORANGE, GREEN, INK = "#0072B2", "#D55E00", "#009E73", "#333333"
MODEL_COLORS = {
    "seasonal-naive": ORANGE, "arima": "#E69F00", "ets": "#CC79A7",
    "chronos2": GREEN, "timesfm25": "#56B4E9", "t0-gamma": BLUE,
}

def new_figure(title, ylabel="Births per day"):
    fig = go.Figure()
    fig.update_layout(
        title=title, yaxis_title=ylabel, hovermode="x unified", height=380,
        legend=dict(orientation="h", y=-0.2), margin=dict(t=60, b=40),
    )
    return fig

def add_line(fig, series, name, color, dash="solid", width=2, markers=False):
    fig.add_scatter(
        x=series.index, y=series.to_numpy(), name=name,
        mode="lines+markers" if markers else "lines",
        line=dict(color=color, dash=dash, width=width),
    )

def mark_cutoff(fig, cutoff):
    fig.add_shape(
        type="line", x0=cutoff, x1=cutoff, y0=0, y1=1, yref="paper",
        line=dict(color=ORANGE, dash="dot"),
    )
    fig.add_annotation(
        x=cutoff, y=1, yref="paper", text="Forecast cutoff", showarrow=False,
        xanchor="right", yanchor="top", font=dict(color="#666666"),
    )

y = load_series()
milestones = {}

def celebrate(key, message):
    milestones[key] = message
    display(HTML(
        f"<progress value='{len(milestones)}' max='4'></progress> "
        f"<b>{len(milestones)}/4 milestones:</b> {message}"
    ))
    print("Completed:", ", ".join(milestones))

> **Curious what a helper does?** Functions like `demo_split` or
> `leaderboard` come from `workshop.py`. In any cell, run
> `help(demo_split)` (or `demo_split?` in Jupyter) to read what it does.
> You never have to edit that file.

## 1. Backtesting: rehearse the future (8 minutes)
One holdout can be lucky. A **rolling-window backtest** repeats:
take the last 168 observed days, forecast 14 days, reveal those outcomes,
move forward 14 days, and repeat. We use three non-overlapping test windows.

```
fold 1: [          168 context days          ][14 test]
fold 2:      [          168 context days          ][14 test]
fold 3:           [          168 context days          ][14 test]
```

Earlier test values can enter later training windows: they would have
become known by that later cutoff. No fold sees its own future.
`strategy="refit"` re-estimates classical models each fold; foundation
models refresh context without training pretrained weights.

### YOUR TURN — Design the splitter (4 minutes)

Use `SlidingWindowSplitter` with a 14-step horizon, 168-day window, and 14-day step. Inspect its three cutoffs before computing any score.

In [ ]:
from sktime.split import SlidingWindowSplitter
cv = None
assert cv is not None, "Set fh, window_length, and step_length."
evaluation_y = benchmark_series(y)
folds = list(cv.split(evaluation_y))
expected_folds = list(splitter().split(evaluation_y))
assert len(folds) == 3, "There should be exactly three backtest origins."
for (train_idx, test_idx), (expected_train, expected_test) in zip(folds, expected_folds):
    np.testing.assert_array_equal(train_idx, expected_train)
    np.testing.assert_array_equal(test_idx, expected_test)
    assert train_idx.max() < test_idx.min()
celebrate("designer", "Your backtest respects time.")

**Hint:** The horizon is `np.arange(1, 15)`. Step length 14 avoids overlapping test windows.

<details><summary>Show solution (copy into the exercise cell when you are ready)</summary>

```python
from sktime.split import SlidingWindowSplitter
cv = SlidingWindowSplitter(
    fh=np.arange(1, 15), window_length=168, step_length=14
)
evaluation_y = benchmark_series(y)
folds = list(cv.split(evaluation_y))
expected_folds = list(splitter().split(evaluation_y))
assert len(folds) == 3
for (train_idx, test_idx), (expected_train, expected_test) in zip(folds, expected_folds):
    np.testing.assert_array_equal(train_idx, expected_train)
    np.testing.assert_array_equal(test_idx, expected_test)
    assert train_idx.max() < test_idx.min()
celebrate("designer", "Your backtest respects time.")
```

</details>

The workshop's own `splitter()` is what every model below is scored with.
Read its description and compare it with the one you just built:

In [ ]:
help(splitter)

In [ ]:
schedule = pd.DataFrame([
    {"fold": number, "context_start": evaluation_y.index[tr[0]],
     "cutoff": evaluation_y.index[tr[-1]], "test_start": evaluation_y.index[te[0]],
     "test_end": evaluation_y.index[te[-1]]}
    for number, (tr, te) in enumerate(folds, start=1)
])
display(schedule)

## 2. Agree on the score before seeing the winner (4 minutes)
**MAE:** `mean(abs(actual - forecast))`, in births/day; lower is better.

**Seasonal MASE:** each fold's MAE divided by the mean absolute **7-day
seasonal difference in that fold's training data**. Scale is learned from
training only. MASE < 1 means error below that in-sample seasonal-difference
scale; it does **not** prove a win over seasonal naive on the test window.
Compare test MAEs directly to answer that question.

We avoid MAPE because it is unstable with near-zero actual values and
can distort comparisons. This dataset is nonzero, but MAE remains easier
to explain. Absolute-error metrics favor medians; point-forecast semantics
can differ across models. RMSE would answer a different, spike-sensitive question.

With equal-size non-overlapping windows, mean fold MAE also equals MAE
over all 42 forecast points. Mean fold MASE averages **fold-specific**
scales. Timing here is a practical cost indicator, not a controlled
hardware benchmark: first-load/download time and warm caches differ.

## 3. One sktime benchmarking path (12 minutes)
`workshop.run_benchmark` calls **`sktime.forecasting.model_evaluation.evaluate`**
for every model, with our splitter, `strategy="refit"`, MAE and seasonal MASE.
It retains actuals/predictions and raises on failure instead of ranking NaNs.
The exact implementation is shown below so it is not a black box.

Classical models:
**seasonal naive** repeats last week;
**ARIMA** here is a fixed SARIMA(1,0,1)(1,0,0)[7] with intercept;
**ETS** has additive trend, damping, and additive 7-day seasonality.
These are sensible teaching settings, not automatically tuned best baselines.

In [ ]:
import inspect
from IPython.display import Code
display(Code(inspect.getsource(run_benchmark), language="python"))

In [ ]:
USE_SAVED_BASELINES = False  # True is the explicit CPU/offline-results fallback
score_parts, prediction_parts, evidence = [], [], []
for name in BASELINES:
    if USE_SAVED_BASELINES:
        scores, predictions, metadata = load_run(ROOT / "artifacts/baselines" / name, y)
        origin = "saved CPU result"
    else:
        scores, predictions = run_benchmark(name, y, cv=cv)
        origin = "live in this runtime"
    score_parts.append(scores)
    prediction_parts.append(predictions)
    evidence.append({"model": name, "status": origin})
print("Three classical models completed three folds each.")
celebrate("referee", "All baseline contestants saw the same history.")

### RUN THIS — add available foundation-model evidence
The default loads instructor rehearsal results, validates the data hash
and scores, and explicitly labels missing models. No missing model is ranked.

Optional: run **one** assigned model below in a separate subprocess.
That invokes the same `run_benchmark` path and releases model memory at exit.
Your own run is saved in `artifacts-live/` and is preferred over the
instructor's in `artifacts/`. Delete its folder to go back to the instructor
result. Do not launch all three simultaneously.

In [ ]:
RUN_ASSIGNED_MODEL = False
ASSIGNED_MODEL = "chronos2"
DEVICE = "cpu"  # "cuda" on a GPU runtime
if RUN_ASSIGNED_MODEL:
    if ASSIGNED_MODEL not in FOUNDATIONS:
        raise ValueError("Choose a foundation model from FOUNDATIONS.")
    live_dir = ROOT / "artifacts-live" / ASSIGNED_MODEL
    if live_dir.exists():
        print(f"{live_dir} already exists; using it. Delete the folder to rerun.")
    else:
        command = [
            sys.executable, "scripts/rehearse.py", "--model", ASSIGNED_MODEL,
            "--device", DEVICE, "--output", str(live_dir),
        ]
        if run(command):
            raise RuntimeError("The model run failed. Read the messages above.")

In [ ]:
# Rebuild from the three baseline parts so rerunning does not duplicate folds.
score_parts = score_parts[:3]
prediction_parts = prediction_parts[:3]
evidence = evidence[:3]
for name in FOUNDATIONS:
    saved, source = ROOT / "artifacts-live" / name, "your own run"
    if not saved.exists():
        saved, source = ROOT / "artifacts" / name, "instructor run"
    if saved.exists():
        scores, predictions, metadata = load_run(saved, y)
        if metadata["model"] != name:
            raise ValueError("Saved model identity does not match its folder.")
        score_parts.append(scores)
        prediction_parts.append(predictions)
        evidence.append({"model": name, "status": f"saved real inference ({source})",
                         "device": metadata["device"],
                         "revision": metadata["checkpoint_revision"]})
    else:
        evidence.append({"model": name, "status": "NOT EVALUATED — no saved run"})
all_scores = pd.concat(score_parts, ignore_index=True)
all_predictions = pd.concat(prediction_parts, ignore_index=True)
display(pd.DataFrame(evidence).fillna("not applicable"))

### YOUR TURN — Build an honest leaderboard (5 minutes)

Group by model and average the fold MAEs; sort lowest first. Then calculate percentage improvement versus seasonal naive. A negative improvement is worse. Do not fill missing models with zero error.

In [ ]:
mean_mae = None
assert mean_mae is not None, "Group all_scores by model, select mae, mean, sort_values."
pd.testing.assert_series_equal(
    mean_mae, all_scores.groupby("model")["mae"].mean().sort_values()
)
improvement_percent = None
assert improvement_percent is not None, "100 * (1 - model MAE / naive MAE)"
np.testing.assert_allclose(
    improvement_percent, 100 * (1 - mean_mae / mean_mae["seasonal-naive"])
)
display(pd.DataFrame({"MAE": mean_mae, "improvement_vs_naive_%": improvement_percent}))
celebrate("investigator", "You measured improvement, not just model size.")

**Hint:** Use `all_scores.groupby('model')['mae'].mean().sort_values()`; normalize by naive.

<details><summary>Show solution (copy into the exercise cell when you are ready)</summary>

```python
mean_mae = all_scores.groupby("model")["mae"].mean().sort_values()
improvement_percent = 100 * (1 - mean_mae / mean_mae["seasonal-naive"])
display(pd.DataFrame({"MAE": mean_mae, "improvement_vs_naive_%": improvement_percent}))
celebrate("investigator", "You measured improvement, not just model size.")
```

</details>

In [ ]:
fig = go.Figure(go.Bar(
    x=improvement_percent.index, y=improvement_percent.to_numpy(),
    marker_color=[MODEL_COLORS.get(m, "#999999") for m in improvement_percent.index],
    text=[f"{v:+.1f}%" for v in improvement_percent], textposition="outside",
))
fig.update_layout(
    title="Improvement over seasonal naive (above 0 = better, below 0 = worse)",
    yaxis_title="Improvement in mean MAE (%)", height=380, margin=dict(t=60),
)
fig.add_hline(y=0, line_color=INK)
fig.show()

## 4. A mean can hide a fragile winner (8 minutes)
Look at each fold, not only the average. Three dependent time-series
origins are not enough for a confident significance claim. The largest
error may coincide with a calendar effect that no model was explicitly told.
Future weather observations would leak; a holiday calendar known at the
forecast date would not.

In [ ]:
display(leaderboard(all_scores).round(2))
fold_errors = all_scores.pivot(index="fold", columns="model", values="mae")
display(fold_errors.round(1))
fig = go.Figure([
    go.Bar(x=[f"Fold {f}" for f in fold_errors.index], y=fold_errors[m], name=m,
           marker_color=MODEL_COLORS.get(m, "#999999"))
    for m in fold_errors.columns
])
fig.update_layout(
    barmode="group", title="Does the winner change by forecast origin?",
    yaxis_title="MAE (births/day)", height=380, margin=dict(t=60),
)
fig.show()

selected_fold = 3  # Try 1 or 2 after inspecting the score table
part = all_predictions.loc[all_predictions["fold"] == selected_fold].copy()
part["date"] = pd.to_datetime(part["date"])
actual = part.drop_duplicates("date").set_index("date")["actual"]
fig = new_figure(f"Fold {selected_fold}: shape, timing, and misses")
add_line(fig, actual, "Actual", INK, width=4)
for name, group in part.groupby("model"):
    add_line(
        fig, group.set_index("date")["prediction"], name,
        MODEL_COLORS.get(name, "#999999"), dash="dash", markers=True,
    )
fig.show()

Do errors grow the further ahead we forecast? Each line averages the
absolute error at day 1, day 2, ... day 14 across the three folds.
Look for a steady climb, then look for a **spike**: what could happen on
one specific day that no model was told about?

In [ ]:
errors = all_predictions.sort_values(["model", "fold", "date"]).copy()
errors["abs_error"] = (errors["prediction"] - errors["actual"]).abs()
errors["days_ahead"] = errors.groupby(["model", "fold"]).cumcount() + 1
by_step = errors.groupby(["model", "days_ahead"])["abs_error"].mean().reset_index()
fig = new_figure("Error by days ahead: a steady climb, or a spike?", ylabel="Mean absolute error (births/day)")
for name, group in by_step.groupby("model"):
    add_line(fig, group.set_index("days_ahead")["abs_error"], name,
             MODEL_COLORS.get(name, "#999999"), markers=True)
fig.update_layout(xaxis_title="Days ahead")
fig.show()

### YOUR TURN — make a recommendation (3 minutes)
Write: **"For this series, horizon, and three cutoffs, I would choose ___
because ___. Before deployment I would test ___."**

Consider accuracy, consistency across folds, setup effort, inference time,
and missing evidence. If no foundation results were available, explicitly
say the foundation-versus-baseline question remains unanswered.

In [ ]:
recommendation = ""
if recommendation.strip():
    celebrate("decision", "You made a qualified, evidence-based recommendation.")
else:
    print("Write your recommendation or discuss it with a partner, then record one sentence.")

## Finish line (90 minutes)
Revisit your opening vote. Did you change your mind? What evidence did it?

**Exit ticket:** name one leakage risk, one useful baseline, and one thing
this experiment cannot establish.

You now know the transferable workflow:
**define decision → inspect history → establish baseline → forecast →
backtest fairly → discuss limitations.**

### Optional after the core
- Add Prophet with the separate dependency and the same benchmark below.
- Evaluate 80% interval coverage and interval width with a wrapper that
  actually supports quantiles. Three folds are not a calibration study.
- Add known-future calendar covariates in a separate experiment; compare
  models on equivalent available information, not targets disguised as X.
- Change horizon/context and declare a **new** protocol; don't tune on
  these folds and call the resulting score an untouched test estimate.
- Read the Kumo Forecast / Relational section in `TUTORIAL.md`.

In [ ]:
RUN_PROPHET = False  # Optional. First install it: uv pip install -r requirements/prophet.txt
if RUN_PROPHET:
    prophet_scores, prophet_predictions = run_benchmark("prophet", y, cv=cv)
    display(leaderboard(pd.concat([all_scores, prophet_scores], ignore_index=True)))